In [1]:
import torch

In [4]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [5]:
import torch.nn as nn


class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # All this are on appropriate devices we don't have to move them as they are constant they are directly stored.
        self.register_buffer("mask", torch.tril(torch.ones(context_length, context_length)))
    
    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_score = queries @ keys.transpose(1, 2)

        mask = torch.tril(torch.ones_like(attn_score))
        attn_score = attn_score.masked_fill(self.mask.bool()[:num_tokens, :num_tokens], float("-inf"))
        attn_weights = torch.softmax(
            attn_score / keys.shape[-1] ** 0.5, dim=-1
        )
        context_vec = attn_weights @ values
        return context_vec

In [6]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )
    
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [ ]:
torch.manual_seed(123)

batch = torch.stack((inputs, inputs), dim=0)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, 2)
context_vec = mha(batch)

In [9]:
context_vec

tensor([[[-0.5507, -0.1728,  0.5135,  0.3893],
         [-0.5041, -0.1662,  0.4724,  0.3621],
         [-0.4293, -0.1551,  0.4040,  0.3176],
         [-0.4533, -0.1548,  0.4293,  0.3242],
         [-0.4213, -0.1501,  0.3836,  0.3539],
         [    nan,     nan,     nan,     nan]],

        [[-0.5507, -0.1728,  0.5135,  0.3893],
         [-0.5041, -0.1662,  0.4724,  0.3621],
         [-0.4293, -0.1551,  0.4040,  0.3176],
         [-0.4533, -0.1548,  0.4293,  0.3242],
         [-0.4213, -0.1501,  0.3836,  0.3539],
         [    nan,     nan,     nan,     nan]]], grad_fn=<CatBackward0>)